# 🚀 Financial RAG Pipeline — Chạy trên Kaggle (GPU T4/P100)
### Cấu trúc Modular 11 File `.py` hoàn chỉnh, ghi bằng `%%writefile` theo chuẩn Production

Notebook này triển khai nguyên vẹn toàn bộ hệ thống Financial RAG:
* **Mỗi file `.py` = 1 nhóm cell** (tạo bằng `%%writefile` và chạy kiểm thử ngay lập tức).
* Hỗ trợ đầy đủ: **Qwen 2.5 3B local**, **PaddleOCR Subprocess Isolation**, **Hybrid Search (ChromaDB + BM25 + RRF)** và **Streamlit UI Workspace 2 cột**.
* Kết nối Internet ra ngoài qua **ngrok tunnel**.

**Lưu ý:** Trước khi chạy, vào **Settings** bên phải Kaggle -> **Accelerator** -> Chọn **GPU T4 x2** (hoặc P100).


## Bước 0 — Cài đặt Package & Khởi tạo Cây Thư Mục


In [ ]:
!pip install -q transformers accelerate bitsandbytes chromadb rank-bm25 pdfplumber pypdfium2 streamlit pyngrok paddleocr
!pip install -q paddlepaddle==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!pip install -q protobuf astor opt_einsum decorator distro
!mkdir -p src/config src/ingestion src/core src/llm src/evaluation data/raw data/vector_db
!touch src/__init__.py src/ingestion/__init__.py src/core/__init__.py src/llm/__init__.py src/evaluation/__init__.py


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7

## File 1 — `src/config.py` (Cấu hình Model & Paths)


In [ ]:
%%writefile src/config.py
import os
from dataclasses import dataclass
import torch


@dataclass
class RAGConfig:
    # --- Cấu hình Đường dẫn Dữ liệu (Paths) ---
    data_raw_dir: str = "./data/raw"
    vector_store_path: str = "./data/vector_db"
    eval_data_path: str = "./data/ground_truth_eval.json"

    # --- Cấu hình Text Chunking & Embedding ---
    max_chunk_chars: int = 1500
    chunk_overlap: int = 100
    embedding_provider: str = "sentence-transformers"
    embedding_model_name: str = "BAAI/bge-m3"
    top_k: int = 4

    # --- Cấu hình LLM ---
    llm_provider: str = "transformers"
    llm_model_name: str = "Qwen/Qwen2.5-3B-Instruct"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    max_new_tokens: int = 512
    temperature: float = 0.1
    torch_dtype: str = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    load_in_4bit: bool = True


default_config = RAGConfig()

# Aliases tương thích ngược
CHROMA_PERSIST_DIR = default_config.vector_store_path
EMBEDDING_MODEL = default_config.embedding_model_name
DATA_RAW_DIR = default_config.data_raw_dir
EVAL_DATA_PATH = default_config.eval_data_path



Overwriting src/config.py


In [ ]:
# Kiểm tra file config
!python -c "from src.config import default_config; print('Config loaded successfully. Device:', default_config.device)"


Config loaded successfully. Device: cuda


## File 2 — `src/llm/base.py` (Abstract Class)


In [ ]:
%%writefile src/llm/base.py
from abc import ABC, abstractmethod
from typing import Optional

class BaseLLM(ABC):
    @abstractmethod
    def generate(
        self,
        prompt: str,
        system_prompt: Optional[str] = None,
        max_new_tokens: Optional[int] = None,
        temperature: Optional[float] = None
    ) -> str:
        pass



Writing src/llm/base.py


## File 3 — `src/llm/providers.py` (Qwen 2.5 3B 4-bit Engine)


In [ ]:
%%writefile src/llm/providers.py
from typing import Optional
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from .base import BaseLLM


class TransformersLLM(BaseLLM):
    def __init__(
        self,
        model_name_or_path: str = "Qwen/Qwen2.5-3B-Instruct",
        device: str = "cuda",
        max_new_tokens: int = 512,
        temperature: float = 0.1,
        torch_dtype: str = "float16",
        load_in_4bit: bool = True
    ):
        self.device = device
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

        compute_dtype = torch.float16 if torch_dtype == "float16" and torch.cuda.is_available() else torch.float32

        quantization_config = None
        if load_in_4bit and device == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_use_double_quant=True
            )

        print(f"[*] Đang nạp mô hình: {model_name_or_path} (4-bit={load_in_4bit})...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            quantization_config=quantization_config,
            device_map="auto" if device == "cuda" else None,
            torch_dtype=compute_dtype if not load_in_4bit else None
        )

        if device == "cpu":
            self.model.to("cpu")

    def generate(
        self,
        prompt: str,
        system_prompt: Optional[str] = None,
        max_new_tokens: Optional[int] = None,
        temperature: Optional[float] = None
    ) -> str:
        sys_msg = system_prompt or "You are a helpful assistant."
        messages = [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        input_len = model_inputs.input_ids.shape[-1]

        target_max_tokens = max_new_tokens or self.max_new_tokens
        target_temp = self.temperature if temperature is None else temperature
        do_sample = target_temp > 0.0

        gen_kwargs = {
            "max_new_tokens": target_max_tokens,
            "pad_token_id": self.tokenizer.eos_token_id,
            "do_sample": do_sample
        }
        if do_sample:
            gen_kwargs["temperature"] = target_temp

        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                **gen_kwargs
            )

        new_tokens = generated_ids[0, input_len:]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return response.strip()



Writing src/llm/providers.py


## File 4 — `src/ingestion/paddle_worker.py` (Tiến trình OCR độc lập)


In [ ]:
%%writefile src/ingestion/paddle_worker.py
import sys
import json
from paddleocr import PaddleOCR


def process_images(image_paths):
    ocr = PaddleOCR(use_angle_cls=True, lang="vi")
    results_map = {}

    for img_path in image_paths:
        try:
            result = ocr.ocr(img_path)
            lines = []
            if result:
                for res in result:
                    if not res:
                        continue
                    if isinstance(res, dict):
                        if "rec_text" in res:
                            lines.append(str(res["rec_text"]))
                        elif "rec_texts" in res:
                            lines.extend([str(t) for t in res["rec_texts"]])
                    elif isinstance(res, (list, tuple)):
                        for line in res:
                            if isinstance(line, dict) and "rec_text" in line:
                                lines.append(str(line["rec_text"]))
                            elif isinstance(line, (list, tuple)) and len(line) >= 2:
                                _, text_payload = line[:2]
                                if isinstance(text_payload, (list, tuple)):
                                    lines.append(str(text_payload[0]))
                                else:
                                    lines.append(str(text_payload))
            results_map[img_path] = "\n".join(lines)
        except Exception as e:
            print(f"[!] Lỗi khi OCR ảnh {img_path}: {e}", file=sys.stderr)
            results_map[img_path] = ""

    return results_map


if __name__ == "__main__":
    if len(sys.argv) >= 3:
        _, json_input_path, json_output_path = sys.argv[:3]

        with open(json_input_path, "r", encoding="utf-8") as f:
            paths = json.load(f)

        output_data = process_images(paths)

        with open(json_output_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False)



Writing src/ingestion/paddle_worker.py


In [ ]:
from src.ingestion.pdf_extractor import extract_document_data

# Thay đường dẫn tới file Techcombank của bạn
sample_pdf = "/content/data/raw/techcombank-vas-bao-cao-tai-chinh-hop-nhat-ye25.pdf"
pages = extract_document_data(sample_pdf)

print(f"-> Số trang bóc tách được: {len(pages)}")
if pages:
    print("Nội dung 100 ký tự đầu trang 1:", repr(pages[0]["text"][:100]))

[!] Phát hiện PDF Scan. Đang gọi PaddleOCR qua tiến trình con độc lập...
[!] Lỗi tiến trình PaddleOCR: /content/src/ingestion/paddle_worker.py:7: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang="vi")
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Traceback (most recent call last):
  File "/content/src/ingestion/paddle_worker.py", line 48, in <module>
    output_data = process_images(paths)
  File "/content/src/ingestion/paddle_worker.py", line 7, in process_images
    ocr = PaddleOCR(use_angle_cls=True, lang="vi")
  File "/usr/local/lib/python3.13/dist-packages/paddleocr/_pipelines/ocr.py", line 173, in __init__
    super().__init__(**base_params)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^

## File 5 — `src/ingestion/pdf_extractor.py` (Trích xuất BCTC & Bảng biểu)


In [ ]:
%%writefile src/ingestion/pdf_extractor.py
import os
import sys
import re
import json
import subprocess
import tempfile
from typing import List, Dict, Any
from PIL import Image
import pdfplumber


def clean_financial_number(text: str) -> str:
    """Chuyển đổi (1.234.567) thành -1.234.567 cho bảng tài chính."""
    if not text:
        return ""
    return re.sub(r"\(\s*([0-9\.,]+)\s*\)", r"-\1", text.strip())


def run_paddle_ocr_batch(pil_images: List[Image.Image]) -> List[str]:
    worker_script = os.path.join(os.path.dirname(__file__), "paddle_worker.py")
    temp_dir = tempfile.mkdtemp()
    image_paths = []

    try:
        for idx, img in enumerate(pil_images):
            img_path = os.path.join(temp_dir, f"page_{idx}.jpg")
            img.convert("RGB").save(img_path, format="JPEG", quality=95)
            image_paths.append(img_path)

        input_json = os.path.join(temp_dir, "input.json")
        output_json = os.path.join(temp_dir, "output.json")

        with open(input_json, "w", encoding="utf-8") as f:
            json.dump(image_paths, f)

        cmd = [sys.executable, worker_script, input_json, output_json]
        proc = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")

        if proc.returncode != 0:
            print(f"[!] Lỗi tiến trình PaddleOCR: {proc.stderr}")
            return ["" for _ in pil_images]

        if not os.path.exists(output_json):
            return ["" for _ in pil_images]

        with open(output_json, "r", encoding="utf-8") as f:
            res_dict = json.load(f)

        return [res_dict.get(p, "") for p in image_paths]
    finally:
        for p in image_paths:
            if os.path.exists(p):
                try:
                    os.remove(p)
                except Exception:
                    pass
        for jf in [os.path.join(temp_dir, "input.json"), os.path.join(temp_dir, "output.json")]:
            if os.path.exists(jf):
                try:
                    os.remove(jf)
                except Exception:
                    pass
        try:
            os.rmdir(temp_dir)
        except Exception:
            pass


def extract_document_data(file_path: str, progress_callback=None) -> List[Dict[str, Any]]:
    file_path_str = str(file_path)
    ext = "." + file_path_str.split(".")[-1].lower() if "." in file_path_str else ""
    pages_data = []

    if ext in [".png", ".jpg", ".jpeg"]:
        if progress_callback:
            progress_callback(1, 1)
        img = Image.open(file_path)
        texts = run_paddle_ocr_batch([img])
        return [{
            "page_number": 1,
            "text": texts[0].strip() if texts else "",
            "tables": []
        }]

    with pdfplumber.open(file_path) as pdf:
        total_pages = len(pdf.pages)
        has_any_text = False

        for idx, page in enumerate(pdf.pages):
            page_num = idx + 1
            if progress_callback:
                progress_callback(page_num, total_pages)

            raw_text = page.extract_text(layout=False) or ""
            if raw_text.strip():
                has_any_text = True

            markdown_tables = []
            try:
                tables = page.extract_tables()
                for tbl in tables:
                    if not tbl or len(tbl) < 2:
                        continue
                    cleaned_rows = [
                        [clean_financial_number(cell).replace("\n", " ").strip() if cell else "-" for cell in row]
                        for row in tbl
                    ]
                    headers = [h if h != "-" else f"Col_{i}" for i, h in enumerate(cleaned_rows[0])]
                    md = "| " + " | ".join(headers) + " |\n"
                    md += "| " + " | ".join(["---"] * len(headers)) + " |\n"
                    for row in cleaned_rows[1:]:
                        md += "| " + " | ".join(row) + " |\n"
                    markdown_tables.append(md)
            except Exception:
                pass

            pages_data.append({
                "page_number": page_num,
                "text": raw_text.strip(),
                "tables": markdown_tables
            })

    if not has_any_text:
        print("[!] Phát hiện PDF Scan. Đang gọi PaddleOCR qua tiến trình con độc lập...")
        import pypdfium2 as pdfium
        pdf_doc = pdfium.PdfDocument(file_path)
        pages_data = []

        try:
            max_scan_pages = min(len(pdf_doc), 25)
            pil_images = []

            for idx in range(max_scan_pages):
                if progress_callback:
                    progress_callback(idx + 1, max_scan_pages)
                page = pdf_doc[idx]
                pil_images.append(page.render(scale=140 / 72).to_pil())

            ocr_results = run_paddle_ocr_batch(pil_images)

            for idx, text in enumerate(ocr_results):
                pages_data.append({
                    "page_number": idx + 1,
                    "text": text.strip(),
                    "tables": []
                })
        finally:
            pdf_doc.close()

    return pages_data

extract_pdf_with_tables = extract_document_data



Writing src/ingestion/pdf_extractor.py


## File 6 — `src/ingestion/chunker.py` (Phân đoạn Ngữ nghĩa)


In [ ]:
%%writefile src/ingestion/chunker.py
from typing import List, Dict, Any


def build_semantic_chunks(
    pages_data: List[Dict[str, Any]],
    meta_info: Dict[str, str],
    max_chunk_chars: int = 1500
) -> List[Dict[str, Any]]:
    chunks = []

    for item in pages_data:
        p_num = item.get("page_number", 0)
        text_content = item.get("text", "").strip()
        tables = item.get("tables", [])

        parts = []
        if text_content:
            parts.append(text_content)
        if tables:
            parts.append("### BẢNG BIỂU TÀI CHÍNH:\n" + "\n\n".join(tables))

        full_content = "\n\n".join(parts)
        if not full_content:
            continue

        paragraphs = full_content.split("\n\n")
        current_chunk = ""

        def add_chunk(content: str):
            chunks.append({
                "content": content.strip(),
                "metadata": {
                    **meta_info,
                    "page_number": p_num
                }
            })

        for para in paragraphs:
            para = para.strip()
            if not para:
                continue

            if len(para) > max_chunk_chars:
                if current_chunk:
                    add_chunk(current_chunk)
                    current_chunk = ""
                add_chunk(para)
                continue

            if len(current_chunk) + len(para) <= max_chunk_chars:
                current_chunk += para + "\n\n"
            else:
                if current_chunk:
                    add_chunk(current_chunk)
                current_chunk = para + "\n\n"

        if current_chunk:
            add_chunk(current_chunk)

    return chunks



Writing src/ingestion/chunker.py


## File 7 — `src/core/contextualizer.py` (Contextual Enrichment)


In [ ]:
%%writefile src/core/contextualizer.py
from typing import List, Dict, Any, Optional
from src.config import default_config

def generate_contextualized_chunks(
    whole_doc: str,
    chunks: List[Dict[str, Any]],
    llm: Optional[Any] = None
) -> List[Dict[str, Any]]:
    """Tối ưu hóa: gắn Header Context tĩnh để không làm quá tải GPU laptop/Kaggle."""
    contextualized_chunks = []

    for chunk in chunks:
        meta = chunk.get("metadata", {})
        comp = meta.get("company_name", "Doanh nghiệp")
        year = meta.get("fiscal_year", "N/A")
        page = meta.get("page_number", 0)

        header_context = f"[Tài liệu: BCTC {comp} ({year}) | Trang: {page}]"
        augmented_content = f"{header_context}\n\n{chunk['content']}"

        contextualized_chunks.append({
            "original_content": chunk["content"],
            "augmented_content": augmented_content,
            "metadata": chunk["metadata"]
        })

    return contextualized_chunks



Writing src/core/contextualizer.py


## File 8 — `src/core/vector_store.py` (ChromaDB + BM25 + RRF)


In [ ]:
%%writefile src/core/vector_store.py
import os
import re
from typing import List, Dict, Any
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

try:
    from src.config import CHROMA_PERSIST_DIR, EMBEDDING_MODEL
except ImportError:
    from src.config import default_config
    CHROMA_PERSIST_DIR = default_config.vector_store_path
    EMBEDDING_MODEL = default_config.embedding_model_name


def simple_tokenize(text: str) -> List[str]:
    return re.findall(r"\w+", text.lower())


class HybridChromaStore:
    def __init__(self, collection_name: str = "financial_reports"):
        os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)
        self.client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
        self.embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=EMBEDDING_MODEL
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_fn
        )
        self.bm25: BM25Okapi = None
        self.corpus_chunks: List[Dict[str, Any]] = []

    def add_chunks(self, chunks: List[Dict[str, Any]]):
        if not chunks:
            return

        ids = []
        documents = []
        metadatas = []

        for idx, c in enumerate(chunks):
            meta = c["metadata"].copy()
            page = meta.get("page_number", meta.get("page", 0))
            company = meta.get("company_name", meta.get("company", "corp"))

            chunk_id = f"{company}_p{page}_idx{idx}_{abs(hash(c['augmented_content'])) % 100000}"
            ids.append(chunk_id)
            documents.append(c["augmented_content"])

            cleaned_meta = {}
            for k, v in meta.items():
                if v is not None and isinstance(v, (str, int, float, bool)):
                    cleaned_meta[k] = v
                else:
                    cleaned_meta[k] = str(v) if v is not None else ""
            cleaned_meta["original_content"] = c.get("original_content", c.get("content", ""))
            metadatas.append(cleaned_meta)

        self.collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas
        )

        self.corpus_chunks.extend(chunks)
        tokenized_corpus = [simple_tokenize(c["augmented_content"]) for c in self.corpus_chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def search(self, query: str, top_k: int = 4) -> List[Dict[str, Any]]:
        chroma_res = self.collection.query(
            query_texts=[query],
            n_results=min(top_k * 4, max(self.collection.count(), 1))
        )

        dense_ranks = {}
        dense_docs = {}
        if chroma_res and chroma_res["documents"] and chroma_res["documents"][0]:
            for rank, doc_id in enumerate(chroma_res["ids"][0]):
                dense_ranks[doc_id] = rank + 1
                dense_docs[doc_id] = {
                    "augmented_content": chroma_res["documents"][0][rank],
                    "original_content": chroma_res["metadatas"][0][rank].get("original_content", ""),
                    "metadata": chroma_res["metadatas"][0][rank]
                }

        sparse_ranks = {}
        if self.bm25 and self.corpus_chunks:
            query_tokens = simple_tokenize(query)
            bm25_scores = self.bm25.get_scores(query_tokens)
            top_bm25_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k * 2]

            for rank, idx in enumerate(top_bm25_indices):
                if bm25_scores[idx] > 0:
                    c = self.corpus_chunks[idx]
                    content_key = c["augmented_content"]
                    sparse_ranks[content_key] = rank + 1

        k_const = 60
        combined_scores = {}

        for doc_id, doc_data in dense_docs.items():
            dense_rank = dense_ranks.get(doc_id, 100)
            content_key = doc_data["augmented_content"]
            sparse_rank = sparse_ranks.get(content_key, 100)

            rrf_score = (1.0 / (k_const + dense_rank)) + (1.0 / (k_const + sparse_rank))
            combined_scores[doc_id] = (rrf_score, doc_data)

        sorted_results = sorted(combined_scores.values(), key=lambda x: x[0], reverse=True)
        return [item[1] for item in sorted_results[:top_k]]



Writing src/core/vector_store.py


## File 9 — `src/core/generator.py` (Inference C-Level & Citation)


In [ ]:
%%writefile src/core/generator.py
from typing import List, Dict, Any, Optional
from src.llm.providers import TransformersLLM
from src.config import default_config

_generator_llm: Optional[TransformersLLM] = None

def get_generator_llm() -> TransformersLLM:
    global _generator_llm
    if _generator_llm is None:
        _generator_llm = TransformersLLM(
            model_name_or_path=default_config.llm_model_name,
            device=default_config.device,
            load_in_4bit=default_config.load_in_4bit,
            max_new_tokens=350,
            temperature=0.1
        )
    return _generator_llm

SYSTEM_PROMPT = """Bạn là trợ lý Chuyên gia Phân tích Tài chính C-Level (Financial Copilot).
Nhiệm vụ của bạn là trả lời các câu hỏi dựa CHÍNH XÁC trên ngữ cảnh (Context) được cung cấp.

Quy tắc:
1. KHÔNG tự suy đoán, KHÔNG bịa đặt số liệu ngoài tài liệu.
2. Với các bảng số liệu, nếu cần tính tỷ lệ (như biên lợi nhuận), hãy giải thích rõ công thức và số liệu thô.
3. Sử dụng thẻ <thinking> trước khi đưa ra câu trả lời chính thức để:
   - Phân tích nguồn thông tin và tính toán.
   - Ghi nhận trang tài liệu.
4. Trả lời rõ ràng, chuyên nghiệp bằng Markdown."""

USER_TEMPLATE = """Dưới đây là tài liệu liên quan:
<context>
{CONTEXT}
</context>

Câu hỏi của người dùng:
<query>
{QUERY}
</query>

Hãy trả lời theo định dạng:
<thinking>
[Phân tích nguồn thông tin và logic tính toán tại đây]
</thinking>

[Nội dung trả lời trực tiếp cho lãnh đạo doanh nghiệp]
"""


def generate_financial_answer(
    query: str,
    retrieved_chunks: List[Dict[str, Any]],
    llm: Optional[TransformersLLM] = None
) -> Dict[str, Any]:
    active_llm = llm or get_generator_llm()

    context_str = ""
    for idx, c in enumerate(retrieved_chunks[:2]):
        meta = c.get("metadata", {})
        page = meta.get("page_number", meta.get("page", "N/A"))
        company = meta.get("company_name", meta.get("company", "N/A"))
        context_str += f"\n--- TÀI LIỆU {idx+1} (Trang {page}, {company}) ---\n"
        raw_content = c.get("augmented_content", c.get("content", ""))[:800]
        context_str += raw_content + "\n"

    user_prompt = USER_TEMPLATE.format(CONTEXT=context_str, QUERY=query)
    full_prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"

    raw_text = active_llm.generate(full_prompt, max_new_tokens=350)

    thinking = ""
    answer = raw_text
    if "<thinking>" in raw_text and "</thinking>" in raw_text:
        parts = raw_text.split("</thinking>")
        thinking = parts[0].replace("<thinking>", "").strip()
        answer = parts[-1].strip()

    return {
        "answer": answer,
        "thinking": thinking,
        "citations": [
            {
                "page": c.get("metadata", {}).get("page_number"),
                "company": c.get("metadata", {}).get("company_name"),
                "snippet": c.get("original_content", c.get("content", ""))[:200] + "..."
            }
            for c in retrieved_chunks[:2]
        ]
    }



Writing src/core/generator.py


## File 10 — `src/evaluation/run_evals.py` (Đo Lường & Đánh Giá Chất Lượng)


In [ ]:
%%writefile src/evaluation/run_evals.py
import os
import json
import re
import pandas as pd
from src.core.vector_store import HybridChromaStore
from src.core.generator import generate_financial_answer
from src.llm.providers import TransformersLLM
from src.config import default_config


def compute_keyword_overlap(prediction: str, ground_truth: str) -> float:
    pred_words = set(re.findall(r"\w+", prediction.lower()))
    gt_words = set(re.findall(r"\w+", ground_truth.lower()))
    if not gt_words:
        return 0.0
    overlap = pred_words.intersection(gt_words)
    return round(len(overlap) / len(gt_words), 2)


def create_sample_ground_truth_if_not_exists(file_path: str = "data/ground_truth_eval.json"):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if not os.path.exists(file_path):
        sample_data = [
            {
                "question": "Tổng thu nhập hoạt động của Techcombank năm 2025 là bao nhiêu?",
                "ground_truth": "Tổng thu nhập hoạt động năm 2025 đạt 53.391.125 triệu đồng."
            }
        ]
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(sample_data, f, ensure_ascii=False, indent=2)


def run_evaluation_pipeline():
    eval_path = "data/ground_truth_eval.json"
    create_sample_ground_truth_if_not_exists(eval_path)

    with open(eval_path, "r", encoding="utf-8") as f:
        qa_pairs = json.load(f)

    print("=== Khởi tạo Vector Store và Generator ===")
    store = HybridChromaStore()

    llm = TransformersLLM(
        model_name_or_path=default_config.llm_model_name,
        device=default_config.device,
        load_in_4bit=default_config.load_in_4bit,
        max_new_tokens=350
    )

    results = []
    print(f"\nĐang đánh giá {len(qa_pairs)} câu hỏi...")

    for idx, item in enumerate(qa_pairs, 1):
        q = item["question"]
        gt = item["ground_truth"]

        retrieved = store.search(q, top_k = 4)
        res = generate_financial_answer(q, retrieved, llm=llm)
        pred_ans = res["answer"]
        score = compute_keyword_overlap(pred_ans, gt)

        results.append({
            "Câu hỏi": q,
            "Đáp án chuẩn (Ground Truth)": gt,
            "Qwen 2.5 Trả lời": pred_ans,
            "Độ khớp": score
        })
        print(f" -> Xong [{idx}/{len(qa_pairs)}] - Điểm: {score}")

    df = pd.DataFrame(results)
    print("\n=== KẾT QUẢ ĐÁNH GIÁ OFFLINE ===")
    print(df[["Câu hỏi", "Độ khớp"]].to_string(index=False))


if __name__ == "__main__":
    run_evaluation_pipeline()



Writing src/evaluation/run_evals.py


## File 11 — `app.py` (Streamlit UI Workspace 2 Cột Đối Chiếu)


In [ ]:
%%writefile app.py
import os
import time
import tempfile
import streamlit as st

from src.ingestion.pdf_extractor import extract_document_data
from src.ingestion.chunker import build_semantic_chunks
from src.core.contextualizer import generate_contextualized_chunks
from src.core.vector_store import HybridChromaStore
from src.core.generator import generate_financial_answer
from src.llm.providers import TransformersLLM
from src.config import default_config

st.set_page_config(
    page_title="Hệ Thống Phân Tích Dữ Liệu Doanh Nghiệp",
    page_icon="📑",
    layout="wide"
)

# Custom Styling cho giao diện
st.markdown("""
<style>
    .info-badge {
        background-color: #1e293b;
        border-left: 4px solid #38bdf8;
        padding: 14px 18px;
        border-radius: 8px;
        margin-bottom: 16px;
        font-size: 14px;
    }
    .source-box {
        background-color: #0f172a;
        border: 1px solid #334155;
        border-radius: 8px;
        padding: 14px;
        margin-bottom: 12px;
        font-size: 13.5px;
        line-height: 1.6;
        color: #e2e8f0;
    }
    .page-tag {
        background-color: #0369a1;
        color: #ffffff;
        font-weight: 600;
        padding: 3px 8px;
        border-radius: 4px;
        font-size: 12px;
        margin-bottom: 6px;
        display: inline-block;
    }
    .metric-tag {
        color: #94a3b8;
        font-size: 12px;
    }
</style>
""", unsafe_allow_html=True)


@st.cache_resource(show_spinner=False)
def get_vector_store():
    return HybridChromaStore()


@st.cache_resource(show_spinner=False)
def get_shared_llm():
    return TransformersLLM(
        model_name_or_path=default_config.llm_model_name,
        device=default_config.device,
        load_in_4bit=default_config.load_in_4bit,
        max_new_tokens=350
    )


vector_store = get_vector_store()
llm = get_shared_llm()

# Khởi tạo trạng thái phiên làm việc (Session State)
if "processed_file_id" not in st.session_state:
    st.session_state["processed_file_id"] = None
if "ready_to_chat" not in st.session_state:
    st.session_state["ready_to_chat"] = False
if "messages" not in st.session_state:
    st.session_state.messages = []

# --- THANH BÊN (SIDEBAR): TỰ ĐỘNG XỬ LÝ KHI CÓ FILE ---
with st.sidebar:
    st.title("📂 Quản Lý Hồ Sơ Tài Liệu")
    st.markdown("---")

    doc_title = st.text_input("Tên đơn vị / Tên tài liệu", value="Báo cáo phân tích")
    doc_period = st.selectbox("Kỳ báo cáo / Phiên bản", ["2025", "2024", "2023", "Bản cập nhật"])
    uploaded_file = st.file_uploader("Tải lên tài liệu (PDF hoặc Ảnh)", type=["pdf", "png", "jpg", "jpeg"])

    # 1. TỰ ĐỘNG KÍCH HOẠT XỬ LÝ KHI PHÁT HIỆN FILE MỚI ĐƯỢC TẢI LÊN
    if uploaded_file is not None:
        file_id = f"{uploaded_file.name}_{uploaded_file.size}"

        # Chỉ xử lý nếu đây là file mới tải lên (tránh xử lý lặp lại khi chat)
        if st.session_state["processed_file_id"] != file_id:
            pdf_bytes = uploaded_file.getvalue()
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                tmp.write(pdf_bytes)
                tmp_path = tmp.name

            try:
                with st.status("⚡ Tự động phân tích và nạp dữ liệu...", expanded=True) as status:
                    pages_data = extract_document_data(tmp_path)

                    if not pages_data or not any(p.get("text", "").strip() for p in pages_data):
                        status.update(label="Không thể xử lý tài liệu", state="error")
                        st.error("⚠️ Định dạng tài liệu không hợp lệ hoặc không có nội dung văn bản.")
                        st.session_state["ready_to_chat"] = False
                        st.stop()

                    meta = {
                        "company_name": doc_title,
                        "fiscal_year": doc_period,
                        "filename": uploaded_file.name
                    }
                    raw_chunks = build_semantic_chunks(pages_data, meta)

                    whole_doc = "\n".join([p["text"] for p in pages_data if p.get("text")])
                    augmented_chunks = generate_contextualized_chunks(whole_doc, raw_chunks, llm=llm)

                    vector_store.add_chunks(augmented_chunks)

                    # Ghi nhớ trạng thái hoàn tất
                    st.session_state["processed_file_id"] = file_id
                    st.session_state["ready_to_chat"] = True
                    st.session_state["active_doc"] = {
                        "name": doc_title,
                        "period": doc_period,
                        "total_pages": len(pages_data),
                        "total_segments": len(augmented_chunks),
                        "file_name": uploaded_file.name
                    }
                    status.update(label="✅ Đã xử lý xong và sẵn sàng đối soát!", state="complete", expanded=False)
                    st.toast("Tài liệu đã sẵn sàng! Khung câu hỏi đã được mở khóa.", icon="🎉")
            except Exception as e:
                st.error("⚠️ Đã có sự cố trong quá trình xử lý file.")
                st.session_state["ready_to_chat"] = False
            finally:
                if os.path.exists(tmp_path):
                    try:
                        os.remove(tmp_path)
                    except Exception:
                        pass
    else:
        # Khi người dùng xóa file
        st.session_state["processed_file_id"] = None
        st.session_state["ready_to_chat"] = False
        if "active_doc" in st.session_state:
            del st.session_state["active_doc"]

# 2. KHÓA / MỞ KHÓA KHUNG HỎI ĐÁP
is_ready = st.session_state.get("ready_to_chat", False)

# Khởi tạo biến lưu câu hỏi trong session
if "current_query" not in st.session_state:
    st.session_state["current_query"] = None

# Khung nhập câu hỏi (Streamlit luôn ghim cố định ở đáy màn hình)
user_input = st.chat_input(
    "Nhập nội dung cần tra cứu từ báo cáo tại đây..." if is_ready else "Vui lòng tải lên tài liệu ở thanh bên trái để mở khóa...",
    disabled=not is_ready
)

# Nếu người dùng vừa gõ câu hỏi mới
if user_input:
    st.session_state["current_query"] = user_input

# ==============================================================================
# TRƯỜNG HỢP A: CHƯA NHẬP PROMPT -> HIỂN THỊ VÙNG KHOANH ĐỎ (HERO BANNER & GỢI Ý)
# ==============================================================================
if not st.session_state["current_query"]:
    st.title("📊 Trợ Lý Phân Tích & Đối Soát Báo Cáo")
    st.caption("Nền tảng tra cứu số liệu tự động, hỗ trợ đối chiếu thông tin minh bạch từ báo cáo gốc.")

    if is_ready and "active_doc" in st.session_state:
        doc = st.session_state["active_doc"]
        st.markdown(f"""
        <div class="info-badge">
            <b>Tài liệu đang mở:</b> {doc['file_name']} | <b>Hồ sơ:</b> {doc['name']} ({doc['period']}) |
            <b>Quy mô:</b> {doc['total_pages']} trang (Đã đồng bộ {doc['total_segments']} phần dữ liệu)
        </div>
        """, unsafe_allow_html=True)
    else:
        st.info("👈 **Hướng dẫn:** Hãy tải lên tài liệu ở cột bên trái. Hệ thống sẽ **tự động xử lý ngay lập tức** và mở khóa khung đặt câu hỏi khi hoàn tất.")

    # Các nút gợi ý câu hỏi nhanh
    if is_ready:
        st.markdown("##### 💡 Gợi ý câu hỏi kiểm tra nhanh:")
        c1, c2, c3, c4 = st.columns(4)
        doc_period = st.session_state.get("active_doc", {}).get("period", "mới nhất")
        doc_name = st.session_state.get("active_doc", {}).get("name", "tài liệu")

        if c1.button("📈 Doanh thu & Thu nhập", use_container_width=True):
            st.session_state["current_query"] = f"Tổng thu nhập hoạt động và kết quả kinh doanh năm {doc_period} của {doc_name} đạt bao nhiêu?"
            st.rerun()
        if c2.button("📉 Chi phí & Vận hành", use_container_width=True):
            st.session_state["current_query"] = f"Tình hình chi phí hoạt động và các khoản chi phí lớn nhất trong kỳ là gì?"
            st.rerun()
        if c3.button("🏦 Tài sản & Dư nợ", use_container_width=True):
            st.session_state["current_query"] = f"Tổng tài sản và quy mô cho vay / đầu tư biến động như thế nào đến cuối kỳ?"
            st.rerun()
        if c4.button("🔍 Đánh giá rủi ro & Lưu ý", use_container_width=True):
            st.session_state["current_query"] = f"Báo cáo có những lưu ý hoặc thuyết minh trọng yếu nào về rủi ro và nợ khó đòi?"
            st.rerun()

# ==============================================================================
# TRƯỜNG HỢP B: ĐÃ CÓ PROMPT -> ẨN TOÀN BỘ VÙNG ĐỎ, CHỈ HIỆN 2 CỘT KẾT QUẢ
# ==============================================================================
else:
    active_query = st.session_state["current_query"]

    # Thanh tiêu đề nhỏ gọn kèm nút hỏi câu khác
    top_col1, top_col2 = st.columns(2)
    with top_col1:
        st.markdown(f"### ❓ **Câu hỏi:** *{active_query}*")
    with top_col2:
        if st.button("🔄 Hỏi câu khác", use_container_width=True):
            st.session_state["current_query"] = None
            st.rerun()

    st.markdown("---")

    # Hiển thị chữ "Đang suy nghĩ..." trong lúc AI đang tìm kiếm và tính toán
    with st.spinner("Đang suy nghĩ..."):
        t_start = time.time()
        retrieved_chunks = vector_store.search(active_query, top_k=4)
        retrieve_duration = time.time() - t_start

        t_gen_start = time.time()
        result = generate_financial_answer(active_query, retrieved_chunks, llm=llm)
        gen_duration = time.time() - t_gen_start

    # HIỂN THỊ CÂU TRẢ LỜI TẬP TRUNG
    st.subheader("💡 Kết Quả Phân Tích")
    st.markdown(f"<span class='metric-tag'>Thời gian phản hồi: {gen_duration:.2f} giây &bull; Đối soát xong trong {retrieve_duration:.2f} giây</span>", unsafe_allow_html=True)
    st.markdown("---")

    # 1. Nội dung câu trả lời phân tích chính (Chiếm trọn vẹn màn hình, dễ đọc)
    st.markdown(result["answer"])

    # 2. Đính kèm căn cứ số liệu ngay dưới câu trả lời
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("##### 📌 Căn cứ dữ liệu & Nguồn trích dẫn:")

    citations = result.get("citations", [])
    has_valid_citation = False
    for cit in citations:
        if cit.get("page"):
            has_valid_citation = True
            st.markdown(f"- **Vị trí Trang {cit['page']}** *({cit.get('company', 'Tài liệu')})*: `{cit['snippet']}`")

    if not has_valid_citation:
        st.caption("*(Dữ liệu được tổng hợp từ các đoạn ngữ cảnh liên quan trong tài liệu)*")

    # 3. Hộp gập/mở đối chiếu nguyên văn (chỉ bung ra khi người dùng cần kiểm chứng kỹ)
    if retrieved_chunks:
        with st.expander("📑 Xem lại nguyên văn các đoạn trích từ tài liệu gốc (Đối chứng)", expanded=False):
            for i, c in enumerate(retrieved_chunks, 1):
                meta = c.get("metadata", {})
                page = meta.get("page_number", meta.get("page", "?"))
                comp = meta.get("company_name", "")
                content = c.get("augmented_content", c.get("content", ""))
                cleaned_display = content.split("]\n\n")[-1] if "]\n\n" in content else content

                st.markdown(f"""
                <div class="source-box">
                    <span class="page-tag">Đoạn đối chiếu {i} &bull; Trang {page} ({comp})</span><br>
                    {cleaned_display}
                </div>
                """, unsafe_allow_html=True)


Writing app.py


## Bước 12 — Khởi chạy Streamlit ngầm & Mở Public URL qua `pyngrok`


In [ ]:
import subprocess, time
from pyngrok import ngrok

# Dán token ngrok của bạn vào đây (Lấy free tại https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = "36ywysaWvIQxgAgCrq2mk80vf8q_2t4mM7q16d4sQF8bDsUgv"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

proc = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501", "--server.headless", "true"
])
time.sleep(8)

public_url = ngrok.connect(8501)
print(f"🚀 Financial RAG Workspace đang chạy tại: {public_url}")
print("Giữ cho cell này tiếp tục chạy (không bấm ⏹) để duy trì kết nối web app.")


🚀 Financial RAG Workspace đang chạy tại: NgrokTunnel: "https://polymorphistic-unshaking-stacee.ngrok-free.dev" -> "http://localhost:8501"
Giữ cho cell này tiếp tục chạy (không bấm ⏹) để duy trì kết nối web app.


---
## Ghi chú Vận hành & Triển khai trên Kaggle
* **GPU Memory**: Trên Kaggle (Tesla T4 16GB VRAM), Qwen 2.5 3B chạy ở chế độ 4-bit chỉ tốn khoảng 2.5GB, bạn có thể tự do tăng `max_new_tokens` lên 512 hoặc 1024 mà không sợ tràn VRAM.
* **Xử lý PDF Scan**: File PDF tải lên qua giao diện sẽ được đọc bằng `pdfplumber` trước; nếu là file scan giấy như Techcombank, hệ thống sẽ tự động chuyển sang `pypdfium2` và `PaddleOCR` trong tiến trình con độc lập.
* **Tải trọn bộ 11 file `.py`**: Toàn bộ các file nằm trong thư mục `/kaggle/working/` có thể được tải về máy bất kỳ lúc nào qua thanh File Explorer bên phải của Kaggle.


In [ ]:
# Xem tiến trình Streamlit và in lỗi ra màn hình
!cat /content/streamlit.log 2>/dev/null || pgrep -a streamlit

3797 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.headless true


In [ ]:
ngrok.kill()